<table  align="left" width="100%"> <tr>
        <td  style="background-color:#ffffff;"><a href="https://qworld.net" target="_blank"><img src="../images/qworld/qworld.jpg" width="35%" align="left"></a></td>
        <td  align="right" style="background-color:#ffffff;vertical-align:bottom;horizontal-align:right">
            prepared by Özlem Salehi Köken
        </td>        
</tr></table>

# Quantum Approximate Optimization Algorithm - Basics and Motivation


Quantum approximate optimization algorithm (QAOA) is a hybrid algorithm for solving combinatorial optimization problems, first introduced by [Farhi et al](https://arxiv.org/abs/1411.4028). In this notebook, we will learn about its basics and motivation.

In combinatorial optimization problems, we look for the best solution among a finite set of solutions. The solution space is often exponentially large. The solution space being finite and discrete makes it harder to find the best solution. For instance, although there are efficient methods for continuous optimization problems which have infinite solution space, many of the combinatorial optimization problems belong to the class NP-hard, i.e., no efficient algorithm is known for solving them.




## How QAOA Works?

QAOA tries to prepare a state whose expectation value with respect to $H_C$ is as low as possible, approximating the ground state.
Note that QAOA is an approximate algorithm and it does not guarantee the optimal solution.

Consider a function $f$ defined on $n$ bits. Suppose that the Hamiltonian $H_C$ satisfies

$$
H_C \ket{x} = f(x) \ket{x},
$$
for $x \in \{0,1\}^n$. The ground state of $H_C$ corresponds to $x$ that minimizes $f(x)$ and the ground state energy is the minimum value of $f(x)$.

Therefore, minimizing the expectation value of $H_C$ is equivalent to minimizing $f(x)$.

We will look at how to express a combinatorial optimization problem as a function of bit strings $f(x)$ and how to obtain the corresponding Hamiltonian $H_C$ in the next notebook. Let us note that such Hamiltonians are always diagonal in the computational basis.  







### Ansatz

In QAOA, the following quantum state serves as our ansatz:

$$\ket{\psi(\gamma, \beta)} = U (H_B , \beta_p)U (H_C , \gamma_p) \dots  U (H_B , \beta_1)U (H_C , \gamma_1)|s⟩,$$

where

$$
U (H_B , \beta_i) = exp(-i \beta_i H_B),
$$

$$
U (H_C , \gamma_i) = exp(-i \gamma_i H_C),
$$

and $p$ is called the number of steps or layers that determines how many times the unitaries $U (H_B , \beta_i)U (H_C , \gamma_i)$ are applied. $H_B$ is the mixer Hamiltonian defined as $-\sum_{i=1}^n X_i$ and  $\ket{s} = \ket{+}^n$ is the initial state which is the ground state of $H_B$. $\beta$ and $\gamma$ form a set of $2p$ parameters.



<figure>
    <img src='../images/qaoa.png' />
</figure>



Overall, we can express the QAOA ansatz as

$$
\prod_{i=1}^p exp(-i \beta_i H_B)exp(-i \gamma_i H_C)\ket{+}^n.
$$

In QAOA, we start with a set of initial parameters, and then iteratively measure the quantum state, and update the parameters with the help of a classical optimizer so that our cost function, which is equal to the expectation value of $H_C$ in this case, is minimized. Note that the parameters $\beta_i$ and $\gamma_i$ determine the duration we evolve the state by the Hamiltonians $H_B$ and $H_C$.



In QAOA as introduced by Farhi et al., $H_B = -\sum_{i=1}^n X_i$ and the initial state is picked as its ground state $\ket{s} = \ket{+}^n$.

In general, one can pick different $H_B$ as long as it does not commute with $H_C$ and pick the initial state accordingly as the ground state of $H_B$. This variant is called Quantum Alternating Operator Ansatz.


To understand how and why QAOA works, we need to understand some concepts.


### Trotterization


We can express $U(T,0)$ as

\begin{align}
U(T,0)&= U (t_n, t_{n-1})U (t_{n-1}, t_{n-2}) \cdots U (t_{1}, t_0)\\
&= \prod_{k=1}^{n} U(t_k, t_{k-1}),
\end{align}
where $t_k = k \delta t$ $ (k=0,1,\dots,n)$, $t_n = T$, $t_0 = 0$, and $\delta t = T/n$. Here, we take small time steps $\delta t$ to discretize the continuous evolution. Note that this step is exact for a time independent Hamiltonian.

### Task 1

For the Hamiltonian $XZ+YZ+IZ$:
- Use diagonalization to compute $U(t,t_0) = exp \left(-iH\cdot(t-t_0)\right)$ for $t_0=0$ and $t=1$.
- Implement the unitary using the formula $\prod_{k=1}^{n} U(t_k, t_{k-1})$ where $t_n = 1$, $t_0=0$, $t_k = kt_n/n$, and $n=10$. Use `HamiltonianGate` to get the corresponding unitary at each time step.

Verify that the two results you obtain are the same.

In [ ]:
import numpy as np
from scipy.linalg import eig
from qiskit.circuit.library import HamiltonianGate
from qiskit.quantum_info import SparsePauliOp
from qiskit.quantum_info import Operator

np.set_printoptions(precision=6, suppress=True)
H =  SparsePauliOp(["XZ","YZ","IZ"], [1,1,1])
t = 1

In [ ]:
### Use diagonalization to compute U1




In [ ]:
### Use the product to compute U2


In [ ]:
assert np.allclose(U1,U2)

[click for our solution](01_qaoa_solutions.ipynb#Task1)



In our case where the Hamiltonian is time-dependent, during the small time step, we assume that the Hamiltonian is approximately constant, and approximate $U(t_k ,t_{k-1})$ as

$$
U(t_k ,t_{k-1}) \approx exp(−i H(t_k)\cdot\delta t),
$$
where $H(t_k)$ is the Hamiltonian at time $t_k$. Hence,

$$
U(T,0) \approx \prod_{k=1}^{n}exp(−i H( k \delta t)\cdot\delta t).
$$


In the case of AQC, recall that the Hamiltonian at time $t$ is given by

$$
H(t)= \left (1- \frac{t}{T}\right ) H_i + \frac{t}{T} H_f.
$$

Replacing this in the above equation we get

$$
U(T,0) \approx \prod_{k=1}^{n}exp \left (−i \left ( \left (1- \frac{k \delta t}{T}\right ) H_i + \frac{k \delta t}{T} H_f \right )   \delta t \right ).
$$

Now we use the *Trotter formula* which states that $e^{x(A+B)} = e^{xA}e^{xB} + O(x^2)$ for non-commuting Hamiltonians $A$ and $B$.

$$
U(T,0) \approx \prod_{k=1}^{n}exp \left (−i \left (1- \frac{k \delta t}{T} \right ) \delta t H_i \right )  exp \left ( −i \frac{k \delta t}{T} \delta t H_f \right ).   
$$




This equation suggests that by applying the Hamiltonians $H_i$ and $H_f$ iteratively, we can approximate adiabatic evolution as $n$ gets larger and larger.

This observation is the inspiration behind QAOA. However, QAOA does not exactly mimic the discretized adiabatic evolution above. Instead of using time-dependent coefficients determined by the adiabatic schedule, QAOA introduces the parameters $\beta$ and $\gamma$ as free variational parameters that are optimized by a classical optimizer.